In [ ]:
import zipfile
import json
import pickle
import numpy as np
from scipy.sparse import csr_matrix, save_npz

# Método para sacar los datos del zip a una matriz con los diccionarios de mapeo

In [ ]:
def zip_to_sparse_matrix(zip_path: str):
    rows = []
    cols = []

    pid_to_idx = {}
    track_to_idx = {}

    current_pid_idx = 0
    current_track_idx = 0

    with zipfile.ZipFile(zip_path, "r") as zipf:
        filenames = [f for f in zipf.namelist() if f.endswith(".json")]
        for i, file in enumerate(filenames):
            with zipf.open(file) as f:
                data = json.loads(f.read())
                for playlist in data["playlists"]:

                    pid = playlist["pid"]

                    # procesamos los pids de las playlist a índices
                    if pid not in pid_to_idx: # realmente no haría falta, pero lo dejamos por si aca
                        pid_to_idx[pid] = current_pid_idx
                        current_pid_idx +=1

                    p_idx = pid_to_idx[pid]

                    # procesamos los tracks_uri a índices
                    tracks = playlist.get("tracks", [])

                    if tracks:
                        for track in tracks:
                            track_uri = track["track_uri"]

                            if track_uri not in track_to_idx:
                                track_to_idx[track_uri]=current_track_idx
                                current_track_idx += 1

                            t_idx = track_to_idx[track_uri]
                            
                            rows.append(p_idx)
                            cols.append(t_idx)

    num_playlist = len(pid_to_idx)
    num_tracks = len(track_to_idx)

    print(f"Número total de playlist: {num_playlist}")
    print(f"Número total de tracks: {num_tracks}")

    rows_np = np.array(rows, dtype = np.int32)
    cols_np = np.array(cols, dtype = np.int32)
    data_np = np.ones(len(rows_np), dtype = np.int8) # también podría ser la longitud de las columnas porque tenemos que colocar tantos 1s como coordenadas haya
                                                     # una coordenada está formada (valor_fila, valor_columna)

    matrix = csr_matrix((data_np, (rows_np, cols_np)),
                        shape = (num_playlist, num_tracks),
                        dtype = np.int8)

    return matrix, pid_to_idx, track_to_idx

# Lo hacemos para test (porque es más pequeñito)

In [16]:
sparse_matrix_test, map_pid_test, map_track_test = zip_to_sparse_matrix("datos/spotify_test_playlists.zip")

Número total de playlist: 10000
Número total de tracks: 111311


### Guardamos la matriz en local

In [17]:
save_npz("matrices/sparse_matrix_test.npz", sparse_matrix_test, compressed=False)

#### podríamos usar pickle en vez de json (pq json transforma las claves a str)

In [ ]:
with open("mapas/pid_test.pkl", "wb") as f:
    pickle.dump(map_pid_test, f)

with open("mapas/tracks_test.pkl", "wb") as f:
    pickle.dump(map_track_test, f)

# Ahora para train

In [35]:
sparse_matrix_train, map_pid_train, map_tracks_train = zip_to_sparse_matrix("datos/spotify_train_dataset.zip")

Número total de playlist: 990000
Número total de tracks: 2262292


In [36]:
save_npz("matrices/sparse_matrix_train.npz", sparse_matrix_train)

In [37]:
with open("mapas/pid_train.pkl", "wb") as f:
    pickle.dump(map_pid_train, f)

with open("mapas/tracks_train.pkl", "wb") as f:
    pickle.dump(map_tracks_train, f)